In [ ]:
%run manage_files

In [ ]:
%run manage_tables

In [ ]:
%run transformation

In [ ]:
%run TaxiFolder

In [ ]:
%run ZoneTable

In [ ]:
from pyspark.sql.functions import *
from abc import ABC, abstractmethod

In [ ]:
class GeneralTaxiTransformer(ABC):

    DATA_FOLDER_TEMPLATE = "Files/NYC_Taxi/"
    @property
    @abstractmethod
    def TABLE_NAME(self) -> str:
        pass

    DEFAULT_SCHEMA = {
        "pickup_datetime": {"type": "timestamp", "nullable": False},
        "dropoff_datetime": {"type": "timestamp", "nullable": False},
        "pickup_location_id": {"type": "integer", "nullable": True},
        "dropoff_location_id": {"type": "integer", "nullable": True}
    }
    @property
    @abstractmethod
    def ADDITIONAL_SCHEMA(self) -> dict:
        pass
    
    DEFAULT_RESULT_COLUMNS = ["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"]
    @property
    @abstractmethod
    def ADDITIONAL_RESULT_COLUMNS(self) -> list:
        pass
    
    DEFAULT_COLUMN_MAPPING = {
        "Trip_Pickup_DateTime": "pickup_datetime",
        "tpep_pickup_datetime": "pickup_datetime",
        "lpep_pickup_datetime": "pickup_datetime",
        "Trip_Dropoff_DateTime": "dropoff_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "dropOff_datetime": "dropoff_datetime",
        "lpep_dropoff_datetime": "dropoff_datetime",
        "PUlocationID": "pickup_location_id",
        "DOlocationID": "dropoff_location_id",
        "PULocationID": "pickup_location_id",
        "DOLocationID": "dropoff_location_id"
    }
    @property
    @abstractmethod
    def ADDITIONAL_COLUMN_MAPPING(self) -> dict:
        pass
    
    TAXI_VALUE_RANGES = {}

    def __init__(self):
        self.folder_path = self.DATA_FOLDER_TEMPLATE + self.TABLE_NAME
        self.table_name = self.TABLE_NAME
        self.schema = {**self.DEFAULT_SCHEMA, **self.ADDITIONAL_SCHEMA}
        self.result_columns = self.DEFAULT_RESULT_COLUMNS + self.ADDITIONAL_RESULT_COLUMNS
        self.column_mapping = {**self.DEFAULT_COLUMN_MAPPING, **self.ADDITIONAL_COLUMN_MAPPING}
        self.date_column = "pickup_datetime"
        self.location_id_columns = ["pickup_location_id", "dropoff_location_id"]

    def fill_empty_numeric(self, df, column_types):
        numeric_columns = [(column, data_type) for column, data_type in column_types.items() if data_type in ["integer", "double"] and column not in self.location_id_columns]
        for column, data_type in numeric_columns:
            if data_type == "double":
                df = df.withColumn(column, nanvl(col(column), lit(0.0)))
        return df.fillna(0, subset=[column for column, _ in numeric_columns])
    
    def filter_year_month(self, df, file_path):
        month = file_path[-15:-8]
        return df.filter(date_format(col(self.date_column), "yyyy-MM") == month)

    def calculate_total(self, df):
        return df

    def filter_taxi_coordinates(self, df):
        return df

    def normalize_taxi(self, df):
        column_types = {column: prop["type"] for column, prop in self.schema.items()}
        df = rename_columns(df, self.column_mapping)
        df = add_missing_columns(df, column_types)
        df = cast_columns(df, column_types)
        df = self.fill_empty_numeric(df, column_types)
        df = self.calculate_total(df)
        df = select_columns(df, column_types)
        return df

    def filter_taxi(self, df, file_path):
        df = drop_null_rows(df, [column for column, prop in self.schema.items() if not prop["nullable"]])
        df = self.filter_taxi_coordinates(df)
        df = filter_column_ranges(df, self.TAXI_VALUE_RANGES)
        df = self.filter_year_month(df, file_path)
        return df
    
    def load_and_process_taxi_table(self, file_path):
        df = read_parquet_file(file_path)
        df = self.normalize_taxi(df)
        df = self.filter_taxi(df, file_path)
        return df
    
    def assign_special_location_ids(self, df, unknown_location_id, outside_nyc_location_id):
        for location_id in self.location_id_columns:
            df = df.withColumn(location_id, when(col(location_id).isNull(), unknown_location_id).otherwise(col(location_id)))
        return df

    def filter_valid_location_ids(self, df, valid_location_ids):
        return df.filter(col(self.location_id_columns[0]).isin(valid_location_ids) & col(self.location_id_columns[1]).isin(valid_location_ids))

    def filter_nyc_trips(self, df, unknown_location_id, outside_nyc_location_id):
        return df.filter(
            (~col("pickup_location_id").isin(unknown_location_id, outside_nyc_location_id)) |
            (~col("dropoff_location_id").isin(unknown_location_id, outside_nyc_location_id))
        )
    
    def finalize_taxi_locations(self, df, unknown_location_id, outside_nyc_location_id, valid_location_ids):
        df = self.assign_special_location_ids(df, unknown_location_id, outside_nyc_location_id)
        df = self.filter_valid_location_ids(df, valid_location_ids)
        df = self.filter_nyc_trips(df, unknown_location_id, outside_nyc_location_id)
        df = select_columns(df, self.result_columns)
        return df
    
    def prepare_sedona_env(self, spark, shapefile = None):
        return None

    def enrich_location(self, df, sedona):
        return df

    def transform(self):
        new_files = TaxiFolder(self.folder_path, self.table_name).get_unloaded_parquet_files(self.date_column)
        if not new_files:
            return
        zone_table = ZoneTable()
        unknown_location_id, outside_nyc_location_id = zone_table.get_special_location_ids()
        valid_location_ids = zone_table.get_valid_location_ids()
        sedona = self.prepare_sedona_env(spark)
        for file_path in new_files:
            df = self.load_and_process_taxi_table(file_path)
            df = self.enrich_location(df, sedona)
            df = self.finalize_taxi_locations(df, unknown_location_id, outside_nyc_location_id, valid_location_ids)
            save_table(df, self.table_name, "Silver_lakehouse", "append")
